# OS & Shell Tutor: RAG Chatbot Notebook

Loads your **saved LoRA adapter** from Drive, indexes your textbook/slides, and launches a chatbot.

**Before you start**
1. `Runtime -> Change runtime type -> T4 GPU`
2. Train and save the adapter first with the training notebook (`MyDrive/os_tutor/os_tutor_v1`).
3. Put your documents (`.pdf`, `.pptx`, `.txt`, `.md`) in `MyDrive/os_tutor/docs/`.
4. Run cells top to bottom. Edit only the **Config** cell.

**Later sessions:** set `REBUILD_INDEX = False` in Config to skip re-embedding your documents.

## 1. Install libraries
If Colab asks you to restart the session afterwards, do it, then continue from the next cell.

In [ ]:
!pip install -q unsloth sentence-transformers chromadb pypdf python-pptx gradio

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Config (edit this cell only)

In [ ]:
ADAPTER_DIR   = "/content/drive/MyDrive/os_tutor/os_tutor_v1"   # saved by the training notebook
DOCS_DIR      = "/content/drive/MyDrive/os_tutor/docs"
CHROMA_DIR    = "/content/drive/MyDrive/os_tutor/chroma_db"

REBUILD_INDEX = True     # True first time and whenever you add/change documents; False afterwards
CHUNK_WORDS   = 300      # words per chunk
CHUNK_OVERLAP = 60       # shared words between neighbouring chunks
TOP_K         = 4        # chunks retrieved per question
MAX_SEQ_LEN   = 4096     # larger than training: prompt now includes retrieved context

## 4. Extract text from your documents
Reads each PDF page / slide and remembers the **file name and page number**, so the tutor can cite sources.
**Check the printed sample**: if it's empty or garbled, the PDF is probably scanned and needs OCR first.

In [ ]:
from pathlib import Path
from pypdf import PdfReader
from pptx import Presentation

def read_pdf(path):
    out = []
    for i, page in enumerate(PdfReader(path).pages, start=1):
        text = page.extract_text() or ""
        if text.strip():
            out.append({"text": text, "source": path.name, "page": i})
    return out

def read_pptx(path):
    out = []
    for i, slide in enumerate(Presentation(path).slides, start=1):
        text = "\n".join(s.text_frame.text for s in slide.shapes if s.has_text_frame)
        if text.strip():
            out.append({"text": text, "source": path.name, "page": i})
    return out

def read_txt(path):
    return [{"text": path.read_text(errors="ignore"), "source": path.name, "page": 1}]

pages = []
if REBUILD_INDEX:
    for f in Path(DOCS_DIR).rglob("*"):
        ext = f.suffix.lower()
        if ext == ".pdf":
            pages += read_pdf(f)
        elif ext == ".pptx":
            pages += read_pptx(f)
        elif ext in (".txt", ".md"):
            pages += read_txt(f)
    print(len(pages), "pages/slides extracted")
    if pages:
        print(pages[0]["text"][:500])
    else:
        print("No text found. Check DOCS_DIR and that your files are not scanned images.")
else:
    print("REBUILD_INDEX is False: skipping extraction.")

## 5. Split into chunks
You can't paste a whole textbook into the prompt, and searching small focused pieces finds better matches. Overlap keeps sentences cut at a boundary from being lost.

In [ ]:
def chunk_text(text, size=CHUNK_WORDS, overlap=CHUNK_OVERLAP):
    words = text.split()
    step = size - overlap
    for i in range(0, max(len(words), 1), step):
        piece = " ".join(words[i:i + size])
        if len(piece.split()) > 30:      # skip tiny fragments
            yield piece

chunks, metas = [], []
for p in pages:
    for piece in chunk_text(p["text"]):
        chunks.append(piece)
        metas.append({"source": p["source"], "page": p["page"]})

print(len(chunks), "chunks")

## 6. Embed and store in ChromaDB
An embedding model turns each chunk into numbers where similar meanings sit close together. The database is saved to Drive, so you embed only once.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")
client = chromadb.PersistentClient(path=CHROMA_DIR)

if REBUILD_INDEX:
    try:
        client.delete_collection("os_notes")      # start fresh
    except Exception:
        pass
    col = client.create_collection("os_notes", metadata={"hnsw:space": "cosine"})

    embeddings = embedder.encode(chunks, batch_size=64, normalize_embeddings=True,
                                 show_progress_bar=True)
    B = 500
    for i in range(0, len(chunks), B):
        col.add(
            ids=[str(j) for j in range(i, min(i + B, len(chunks)))],
            documents=chunks[i:i + B],
            embeddings=embeddings[i:i + B].tolist(),
            metadatas=metas[i:i + B],
        )
else:
    col = client.get_collection("os_notes")

print(col.count(), "chunks in the database")

## 7. Test retrieval (before involving the model)
If the right passages don't come back here, the final answer won't be good either.
Fixes: smaller chunks, check extraction quality, or raise `TOP_K`.

In [ ]:
def retrieve(question, k=TOP_K):
    # bge models work best with this prefix on the *query* only
    q = "Represent this sentence for searching relevant passages: " + question
    q_emb = embedder.encode([q], normalize_embeddings=True).tolist()
    res = col.query(query_embeddings=q_emb, n_results=k)
    return list(zip(res["documents"][0], res["metadatas"][0], res["distances"][0]))

for doc, meta, dist in retrieve("What are the four conditions for deadlock?"):
    print(f"[{meta['source']} p.{meta['page']}]  distance={dist:.3f}")
    print(doc[:200], "...\n")

## 8. Load your fine-tuned model from Drive

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_DIR,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

## 9. The RAG loop: retrieve -> put chunks in the prompt -> generate
Your fine-tune was trained without context in the prompt, so it may sometimes ignore the context. If so, make the system prompt firmer or add training examples in the "Context / Question" format.

In [ ]:
SYSTEM = ("You are an OS and shell tutor. Answer using the provided context. "
          "If the context does not contain the answer, say so, then answer from "
          "general knowledge and mark it clearly. Cite sources like (file, p.X). "
          "Explain step by step and be accurate.")

def rag_answer(question, k=TOP_K, max_new_tokens=512, show_sources=True):
    hits = retrieve(question, k)
    context = "\n\n".join(f"[{m['source']}, p.{m['page']}]\n{d}" for d, m, _ in hits)
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         temperature=0.7, top_p=0.8, top_k=20, do_sample=True)
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    if show_sources:
        srcs = sorted({f"{m['source']} p.{m['page']}" for _, m, _ in hits})
        answer += "\n\n---\n*Sources retrieved:* " + ", ".join(srcs)
    return answer

print(rag_answer("What are the four conditions for deadlock?"))

## 10. Launch the chatbot
`share=True` gives a temporary public link that works while this Colab session is running.
Gradio passes the conversation history, so the last few turns are added to the prompt for follow-up questions.

In [ ]:
import gradio as gr

def chat(message, history):
    return rag_answer(message)

gr.ChatInterface(
    chat,
    title="OS & Shell Tutor",
    description="Ask about processes, scheduling, memory, file systems, and shell commands.",
    examples=["What are the four conditions for deadlock?",
              "How does round robin scheduling work?",
              "How do I find all .log files larger than 10MB?"],
).launch(share=True)

## Troubleshooting
- **Empty text from PDFs:** scanned document, so run OCR (`ocrmypdf`) first.
- **Irrelevant chunks:** try `CHUNK_WORDS = 200`, a larger `TOP_K`, or add a reranker.
- **Out of memory:** lower `TOP_K` to 3 or `MAX_SEQ_LEN` to 3072; make sure no other notebook is using the GPU.
- **Adapter not found:** check that `ADAPTER_DIR` matches the folder the training notebook saved to.
- **Copyright:** fine for private study, but check licensing before publicly sharing an app that contains copyrighted material.